# Проверка: клиенты торгового эквайринга, май–июль 2026

Скрипт: `sources/sql/trade_acquiring_actual_clients.sql` (задача 12554).

Здесь не выгружается вся таблица с ЦФТ и РФ. Считается периметр секции 01 `final_df`: договоры SA за месяц.

**Клиент** — уникальный ИНН. **Договор** — `n_agr`. У одного ИНН может быть несколько договоров.

Месяцы: май, июнь, июль 2026. Договор входит в месяц, если `d_valid_from <= month_end` и (`d_valid_to` пустая или `>= month_start`), и есть неудалённая привязка `agr_terms` с `cf_ter_type = 'P'`, которая пересекает месяц: `d_valid_from <= month_end` и (`d_valid_to` пустая или `> month_start`). У привязки сравнение строгое: `>`, не `>=`.

`cf_ter_type = 'P'` — тот же входной фильтр, что в секции 01. Вторая ячейка показывает остальные коды за июль и сколько договоров они держат.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.0f}'.replace(',', ' '))

LAKE_USER = 'Shestopalov-VYur'
LAKE_PASSWORD = None
for cand in [
    Path.cwd(),
    Path.cwd() / 'sources' / 'sql',
    Path('/home/jovyan/documents/Equaring'),
    Path('/home/jovyan/documents/Equaring/sources/sql'),
]:
    if (cand / 'connection_secrets.py').exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        from connection_secrets import LAKE_USER, LAKE_PASSWORD
        break

user_params = {'user_name': LAKE_USER}
if LAKE_PASSWORD:
    user_params['password'] = LAKE_PASSWORD

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params=user_params,
)
imp._init_connection()
print('imp ready, user =', LAKE_USER)


In [ ]:
MONTHS = [
    ('2026-05', '2026-05-01', '2026-05-31'),
    ('2026-06', '2026-06-01', '2026-06-30'),
    ('2026-07', '2026-07-01', '2026-07-31'),
]
MEM_LIMIT = '16g'

months_sql = '\n  union all\n'.join(
    "  select '{label}' as report_month, cast('{month_start}' as date) as month_start, cast('{month_end}' as date) as month_end".format(
        label=label, month_start=month_start, month_end=month_end,
    )
    for label, month_start, month_end in MONTHS
)

sql_counts = f"""
with months as (
{months_sql}
),
sa_open as (
  select
    m.report_month,
    cast(a.n_agr as string) as n_agr,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
  from months m
  join ods_alpha.scd1_agreements a
    on cast(a.d_valid_from as date) <= m.month_end
   and (a.d_valid_to is null or cast(a.d_valid_to as date) >= m.month_start)
  join ods_alpha.scd1_companies c
    on c.n_cmp = a.n_cmp_client
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and coalesce(a.ods_deleted_flg, '0') <> '1'
    and coalesce(c.ods_deleted_flg, '0') <> '1'
    and c.c_inn is not null
),
terms_p as (
  select distinct
    m.report_month,
    cast(t.n_agr as string) as n_agr
  from months m
  join ods_alpha.scd1_agr_terms t
    on cast(t.d_valid_from as date) <= m.month_end
   and (t.d_valid_to is null or cast(t.d_valid_to as date) > m.month_start)
  where upper(trim(cast(t.cf_ter_type as string))) = 'P'
    and coalesce(t.ods_deleted_flg, '0') <> '1'
)
select
  s.report_month,
  count(distinct case when s.inn <> '' then s.inn end) as clients_sa_open,
  count(distinct s.n_agr) as contracts_sa_open,
  count(distinct case when tp.n_agr is not null and s.inn <> '' then s.inn end) as clients_with_p,
  count(distinct case when tp.n_agr is not null then s.n_agr end) as contracts_with_p
from sa_open s
left join terms_p tp
  on tp.report_month = s.report_month
 and tp.n_agr = s.n_agr
group by s.report_month
order by s.report_month
"""

with imp:
    imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
    counts = imp.fetch(sql_counts)

counts = counts.copy()
for col in ['clients_sa_open', 'contracts_sa_open', 'clients_with_p', 'contracts_with_p']:
    counts[col] = pd.to_numeric(counts[col], errors='coerce').fillna(0).astype('int64')

counts['clients_dropped_by_p'] = counts['clients_sa_open'] - counts['clients_with_p']
counts['contracts_dropped_by_p'] = counts['contracts_sa_open'] - counts['contracts_with_p']

show = counts.rename(columns={
    'report_month': 'Месяц',
    'clients_with_p': 'Клиенты (ИНН), как final_df',
    'contracts_with_p': 'Договоры, как final_df',
    'clients_sa_open': 'Клиенты SA без фильтра P',
    'contracts_sa_open': 'Договоры SA без фильтра P',
    'clients_dropped_by_p': 'Клиенты отсечены фильтром P',
    'contracts_dropped_by_p': 'Договоры отсечены фильтром P',
})
show = show[[
    'Месяц',
    'Клиенты (ИНН), как final_df',
    'Договоры, как final_df',
    'Клиенты SA без фильтра P',
    'Договоры SA без фильтра P',
    'Клиенты отсечены фильтром P',
    'Договоры отсечены фильтром P',
]]
show


## Какие коды `cf_ter_type` есть у договоров SA за июль 2026

Те же границы месяца, что в секции 01: договор пересекает июль, привязка пересекает июль (`d_valid_to > 2026-07-01`). `P` остаётся в периметре. Остальные коды нет.


In [ ]:
MONTH_START = '2026-07-01'
MONTH_END = '2026-07-31'

sql_types = f"""
with sa_open as (
  select cast(a.n_agr as string) as n_agr
  from ods_alpha.scd1_agreements a
  join ods_alpha.scd1_companies c
    on c.n_cmp = a.n_cmp_client
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and cast(a.d_valid_from as date) <= cast('{MONTH_END}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{MONTH_START}' as date))
    and coalesce(a.ods_deleted_flg, '0') <> '1'
    and coalesce(c.ods_deleted_flg, '0') <> '1'
    and c.c_inn is not null
)
select
  coalesce(nullif(upper(trim(cast(t.cf_ter_type as string))), ''), '(пусто)') as cf_ter_type,
  count(distinct cast(t.n_agr as string)) as contracts,
  count(*) as term_rows
from ods_alpha.scd1_agr_terms t
join sa_open s
  on s.n_agr = cast(t.n_agr as string)
where cast(t.d_valid_from as date) <= cast('{MONTH_END}' as date)
  and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{MONTH_START}' as date))
  and coalesce(t.ods_deleted_flg, '0') <> '1'
group by 1
order by contracts desc
"""

with imp:
    imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
    types = imp.fetch(sql_types)

types = types.copy()
types['contracts'] = pd.to_numeric(types['contracts'], errors='coerce').fillna(0).astype('int64')
types['term_rows'] = pd.to_numeric(types['term_rows'], errors='coerce').fillna(0).astype('int64')
types['in_script'] = types['cf_ter_type'].eq('P')
types
